In [ ]:

# Cell 1: Installing the required libraries

!pip -q install requests pandas tqdm


In [ ]:

# Cell 2: Entering my API key

from getpass import getpass

OPENALEX_API_KEY = getpass("Enter your OpenAlex API key: ")

if not OPENALEX_API_KEY.strip():
    raise ValueError("Please enter a valid API key.")

print("API key received.")



Enter your OpenAlex API key: ··········
API key received.


In [ ]:

# Cell 3: Testing OpenAlex and reconstructing an abstract

import requests

BASE_URL = "https://api.openalex.org/works"

def reconstruct_abstract(inverted_index):
    """Convert OpenAlex's abstract index into readable text."""
    if not inverted_index:
        return None

    words = []

    for word, positions in inverted_index.items():
        for position in positions:
            words.append((position, word))

    words.sort(key=lambda item: item[0])

    return " ".join(word for position, word in words)


params = {
    "api_key": OPENALEX_API_KEY,
    "search": "solar energy",
    "filter": "has_abstract:true,language:en,type:article",
    "per_page": 5,
    "select": "id,doi,title,publication_year,abstract_inverted_index"
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=60
)

response.raise_for_status()
data = response.json()

print("Papers returned:", len(data["results"]))

for paper in data["results"]:
    abstract = reconstruct_abstract(
        paper.get("abstract_inverted_index")
    )

    print("\nTitle:", paper.get("title"))
    print("Year:", paper.get("publication_year"))
    print("Abstract:", (abstract or "No abstract")[:500])
    print("-" * 60)


Papers returned: 5

Title: Photochemical Conversion of Solar Energy
Year: 2007
Abstract: Energy is the most important issue of the 21st century. About 85% of our energy comes from fossil fuels, a finite resource unevenly distributed beneath the Earth's surface. Reserves of fossil fuels are progressively decreasing, and their continued use produces harmful effects such as pollution that threatens human health and greenhouse gases associated with global warming. Prompt global action to solve the energy crisis is therefore needed. To pursue such an action, we are urged to save energy a
------------------------------------------------------------

Title: The momentum of the solar energy transition
Year: 2023
Abstract: Decarbonisation plans across the globe require zero-carbon energy sources to be widely deployed by 2050 or 2060. Solar energy is the most widely available energy resource on Earth, and its economic attractiveness is improving fast in a cycle of increasing investments. Here we

In [ ]:

# CELL 4: Collecting 6,000 papers using OpenAlex topic metadata

import time
import requests
import pandas as pd
from tqdm.auto import tqdm

BASE_URL = "https://api.openalex.org"
TARGET_PER_CLASS = 2000
BATCH_SIZE = 100


API_PARAMS = {}
if "OPENALEX_API_KEY" in globals() and OPENALEX_API_KEY.strip():
    API_PARAMS["api_key"] = OPENALEX_API_KEY.strip()

session = requests.Session()

def get_json(url, params, attempts=5):
    """Request data with retries for temporary errors."""
    for attempt in range(attempts):
        try:
            response = session.get(
                url, params=params, timeout=60
            )

            if response.status_code == 429:
                time.sleep(5 * (attempt + 1))
                continue

            response.raise_for_status()
            return response.json()

        except requests.RequestException as error:
            if attempt == attempts - 1:
                raise

            print("Retrying after error:", error)
            time.sleep(3 * (attempt + 1))

    raise RuntimeError("Could not retrieve data.")


def get_entity_number(entity_id):
    """Extract the numeric ID from an OpenAlex entity URL."""
    return str(entity_id).rstrip("/").split("/")[-1]


def reconstruct_abstract(index):
    """Convert an OpenAlex inverted index into readable text."""
    if not index:
        return None

    words = []

    for word, positions in index.items():
        for position in positions:
            words.append((position, word))

    words.sort(key=lambda x: x[0])
    return " ".join(word for position, word in words)


# -------------------------------------------------
# A. Discovering the domain and field IDs
# -------------------------------------------------

domains_data = get_json(
    f"{BASE_URL}/domains",
    {**API_PARAMS, "per_page": 100}
)

domain_ids = {
    item["display_name"]: get_entity_number(item["id"])
    for item in domains_data["results"]
}

print("OpenAlex domains:")
for name, entity_id in domain_ids.items():
    print(name, "->", entity_id)


fields_data = get_json(
    f"{BASE_URL}/fields",
    {**API_PARAMS, "per_page": 100}
)

all_fields = fields_data["results"]

# Fields used for the Business class.
business_keywords = [
    "business",
    "management and accounting",
    "economics",
    "finance"
]

business_fields = [
    field for field in all_fields
    if any(
        keyword in field["display_name"].lower()
        for keyword in business_keywords
    )
]

business_field_ids = {
    get_entity_number(field["id"])
    for field in business_fields
}

print("\nBusiness-related fields discovered:")
for field in business_fields:
    print("-", field["display_name"])


# -------------------------------------------------
# B. Seting up a clean dataset
# -------------------------------------------------

records = []
seen_ids = set()

OUTPUT_FILE = "research_abstracts_6000_topic_based.csv"
CHECKPOINT_FILE = "research_abstracts_topic_checkpoint.csv"

COLUMNS = [
    "title",
    "abstract",
    "label",
    "source_url",
    "year",
    "openalex_id",
    "primary_topic",
    "subfield",
    "field",
    "domain",
    "topic_score"
]


def current_count(label):
    return sum(row["label"] == label for row in records)


# -------------------------------------------------
# C. Collecting papers from a specified OpenAlex filter
# -------------------------------------------------

def collect_category(label, filter_value, skip_business=False):
    global records, seen_ids

    count = current_count(label)

    if count >= TARGET_PER_CLASS:
        return

    print(f"\nCollecting {label}: {count}/{TARGET_PER_CLASS}")

    cursor = "*"

    while count < TARGET_PER_CLASS:

        params = {
            **API_PARAMS,
            "filter": (
                "has_abstract:true,"
                "language:en,"
                "type:article,"
                + filter_value
            ),
            "per_page": BATCH_SIZE,
            "cursor": cursor,
            "select": (
                "id,doi,title,publication_year,"
                "abstract_inverted_index,primary_topic"
            )
        }

        data = get_json(f"{BASE_URL}/works", params)

        papers = data.get("results", [])

        if not papers:
            print("No more papers available for this filter.")
            break

        added = 0

        for paper in papers:

            paper_id = paper.get("id")
            title = paper.get("title")
            abstract = reconstruct_abstract(
                paper.get("abstract_inverted_index")
            )

            if not paper_id or paper_id in seen_ids:
                continue

            if not title or not abstract:
                continue

            # Excluding extremely short abstracts.
            if len(abstract.split()) < 30:
                continue

            topic = paper.get("primary_topic") or {}
            field_info = topic.get("field") or {}
            domain_info = topic.get("domain") or {}
            subfield_info = topic.get("subfield") or {}

            field_name = field_info.get("display_name", "")
            domain_name = domain_info.get("display_name", "")
            field_id = (
                get_entity_number(field_info["id"])
                if field_info.get("id") else ""
            )

            # Validating the requested class against metadata.
            if label == "Science":
                if domain_name not in [
                    "Physical Sciences",
                    "Life Sciences",
                    "Health Sciences"
                ]:
                    continue

            elif label == "Business":
                if field_id not in business_field_ids:
                    continue

            elif label == "Social Science":
                if domain_name != "Social Sciences":
                    continue

                # Business fields are assigned to Business,
                # not Social Science.
                if skip_business and field_id in business_field_ids:
                    continue

            # Prefer a publisher DOI
            source_url = paper.get("doi") or paper_id

            records.append({
                "title": title.strip(),
                "abstract": abstract.strip(),
                "label": label,
                "source_url": source_url,
                "year": paper.get("publication_year"),
                "openalex_id": paper_id,
                "primary_topic": topic.get("display_name"),
                "subfield": subfield_info.get("display_name"),
                "field": field_name,
                "domain": domain_name,
                "topic_score": topic.get("score")
            })

            seen_ids.add(paper_id)
            count += 1
            added += 1

            if count >= TARGET_PER_CLASS:
                break

        # Save progress after each page.
        pd.DataFrame(records, columns=COLUMNS).to_csv(
            CHECKPOINT_FILE,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"{label}: {count}/{TARGET_PER_CLASS} "
            f"(added {added} this page)"
        )

        next_cursor = data.get("meta", {}).get("next_cursor")

        if not next_cursor or next_cursor == cursor:
            break

        cursor = next_cursor
        time.sleep(0.2)

    print(f"Finished {label}: {count}/{TARGET_PER_CLASS}")


# -------------------------------------------------
# D. Collecting Science from three broad domains
# -------------------------------------------------

science_domains = [
    "Physical Sciences",
    "Life Sciences",
    "Health Sciences"
]

for domain_name in science_domains:
    if domain_name not in domain_ids:
        print("Domain not found:", domain_name)
        continue

    if current_count("Science") >= TARGET_PER_CLASS:
        break

    domain_id = domain_ids[domain_name]

    collect_category(
        "Science",
        f"primary_topic.domain.id:{domain_id}"
    )


# -------------------------------------------------
# E. Collecting Business from relevant fields
# -------------------------------------------------

for field in business_fields:
    if current_count("Business") >= TARGET_PER_CLASS:
        break

    field_id = get_entity_number(field["id"])

    collect_category(
        "Business",
        f"primary_topic.field.id:{field_id}"
    )


# -------------------------------------------------
# F. Collecting Social Science, excluding Business fields
# -------------------------------------------------

if "Social Sciences" in domain_ids:
    collect_category(
        "Social Science",
        f"primary_topic.domain.id:{domain_ids['Social Sciences']}",
        skip_business=True
    )


# -------------------------------------------------
# G. Saving the final result
# -------------------------------------------------

df = pd.DataFrame(records, columns=COLUMNS)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nCollection finished.")
print("Total papers collected:", len(df))
print("\nCounts per class:")
print(
    df["label"].value_counts().reindex(
        ["Science", "Business", "Social Science"],
        fill_value=0
    )
)

print("\nSaved to:", OUTPUT_FILE)


OpenAlex domains:
Physical Sciences -> 3
Social Sciences -> 2
Health Sciences -> 4
Life Sciences -> 1

Business-related fields discovered:
- Business, Management and Accounting
- Economics, Econometrics and Finance

Science: 97/2000 (added 97 this page)
Science: 188/2000 (added 91 this page)
Science: 282/2000 (added 94 this page)
Science: 378/2000 (added 96 this page)
Science: 472/2000 (added 94 this page)
Science: 565/2000 (added 93 this page)
Science: 658/2000 (added 93 this page)
Science: 753/2000 (added 95 this page)
Science: 848/2000 (added 95 this page)
Science: 946/2000 (added 98 this page)
Science: 1044/2000 (added 98 this page)
Science: 1141/2000 (added 97 this page)
Science: 1234/2000 (added 93 this page)
Science: 1332/2000 (added 98 this page)
Science: 1425/2000 (added 93 this page)
Science: 1521/2000 (added 96 this page)
Science: 1618/2000 (added 97 this page)
Science: 1713/2000 (added 95 this page)
Science: 1810/2000 (added 97 this page)
Science: 1908/2000 (added 98 this p

In [ ]:

# CELL 5: Verifying category balance and data quality

print("Total records:", len(df))
print("Unique paper IDs:", df["openalex_id"].nunique())

print("\nClass distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df[["title", "abstract", "label"]].isnull().sum())

print("\nDuplicate abstracts:", df["abstract"].duplicated().sum())

print("\nSample records:")
display(
    df[[
        "title", "label", "primary_topic",
        "field", "domain"
    ]].head(15)
)

expected = {
    "Science": 2000,
    "Business": 2000,
    "Social Science": 2000
}

counts = df["label"].value_counts().to_dict()

for label, target in expected.items():
    actual = counts.get(label, 0)
    print(f"{label}: {actual}/{target}")


Total records: 6000
Unique paper IDs: 6000

Class distribution:
label
Science           2000
Business          2000
Social Science    2000
Name: count, dtype: int64

Missing values:
title       0
abstract    0
label       0
dtype: int64

Duplicate abstracts: 28

Sample records:


,title,label,primary_topic,field,domain
0,Radiation Resistant Camera System for Monitori...,Science,Magnetic confinement fusion research,Physics and Astronomy,Physical Sciences
1,Generalized Gradient Approximation Made Simple,Science,Advanced Chemical Physics Studies,Physics and Astronomy,Physical Sciences
2,Efficient iterative schemes for ab initio tota...,Science,Advanced Chemical Physics Studies,Physics and Astronomy,Physical Sciences
3,ImageNet classification with deep convolutiona...,Science,Advanced Neural Network Applications,Computer Science,Physical Sciences
4,Controlling the False Discovery Rate: A Practi...,Science,Statistical Methods in Clinical Trials,Mathematics,Physical Sciences
5,Density-functional thermochemistry. III. The r...,Science,Advanced Chemical Physics Studies,Physics and Astronomy,Physical Sciences
6,Long Short-Term Memory,Science,Neural Networks and Applications,Computer Science,Physical Sciences
7,Development of the Colle-Salvetti correlation-...,Science,Advanced Chemical Physics Studies,Physics and Astronomy,Physical Sciences
8,Projector augmented-wave method,Science,Advanced Chemical Physics Studies,Physics and Astronomy,Physical Sciences
9,The moderator-mediator variable distinction in...,Science,Advanced Causal Inference Techniques,Mathematics,Physical Sciences


Science: 2000/2000
Business: 2000/2000
Social Science: 2000/2000


In [ ]:

# Finding all rows with duplicate abstract text
duplicate_rows = df[
    df["abstract"].duplicated(keep=False)
].sort_values("abstract")

print("Rows involved in duplicates:", len(duplicate_rows))
print("Distinct duplicated abstracts:",
      duplicate_rows["abstract"].nunique())

display(
    duplicate_rows[
        ["title", "label", "abstract", "openalex_id"]
    ].head(10)
)


Rows involved in duplicates: 55
Distinct duplicated abstracts: 27


,title,label,abstract,openalex_id
2949,Dynamic Version of the Economic Lot Size Model,Business,A forward algorithm for a solution to the foll...,https://openalex.org/W2107857629
2937,Dynamic Version of the Economic Lot Size Model,Business,A forward algorithm for a solution to the foll...,https://openalex.org/W4256185527
4076,The Structure of Scientific Revolutions,Social Science,A good book may have the power to change the w...,https://openalex.org/W2752443224
4031,The Structure of Scientific Revolutions,Social Science,A good book may have the power to change the w...,https://openalex.org/W2753533763
3692,Social Marketing: An Approach to Planned Socia...,Business,Can marketing concepts and techniques be effec...,https://openalex.org/W4243443132
3206,Social Marketing: An Approach to Planned Socia...,Business,Can marketing concepts and techniques be effec...,https://openalex.org/W2059518795
3987,Cognition and Corporate Governance: Understand...,Business,"Daniel P. Forbes, Frances J. Milliken, Cogniti...",https://openalex.org/W2116300149
3401,Cognition and Corporate Governance: Understand...,Business,"Daniel P. Forbes, Frances J. Milliken, Cogniti...",https://openalex.org/W4249624191
4436,The Logic of Scientific Discovery.,Social Science,Described by the philosopher A.J. Ayer as a wo...,https://openalex.org/W3127702745
5981,The Logic of Scientific Discovery.,Social Science,Described by the philosopher A.J. Ayer as a wo...,https://openalex.org/W3021587765


In [ ]:

# Removing missing abstracts and exact duplicate abstracts
df["abstract"] = df["abstract"].str.strip()

df = df.dropna(subset=["abstract", "label"])

df = df.drop_duplicates(
    subset=["abstract"],
    keep="first"
).reset_index(drop=True)

print("Records remaining:", len(df))
print("\nCategory distribution:")
print(df["label"].value_counts())


Records remaining: 5972

Category distribution:
label
Science           1994
Business          1991
Social Science    1987
Name: count, dtype: int64


In [ ]:

df.to_csv(
    "research_abstracts_6000_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned dataset saved successfully.")


Cleaned dataset saved successfully.


In [ ]:

import pandas as pd

df = pd.read_csv("research_abstracts_6000_cleaned.csv")

# Remove whitespace and exact duplicate abstracts
df["abstract"] = df["abstract"].str.strip()
df = df.dropna(subset=["abstract", "label"])
df = df.drop_duplicates(subset=["abstract"]).reset_index(drop=True)

print("Unique abstracts:", len(df))
print("\nCategory counts:")
print(df["label"].value_counts())


Unique abstracts: 5972

Category counts:
label
Science           1994
Business          1991
Social Science    1987
Name: count, dtype: int64


In [ ]:

import time
import requests
import pandas as pd

BASE_URL = "https://api.openalex.org"
TARGET = 2000
OUTPUT_FILE = "research_abstracts_6000_cleaned.csv"

session = requests.Session()

api_params = {}
if "OPENALEX_API_KEY" in globals() and OPENALEX_API_KEY.strip():
    api_params["api_key"] = OPENALEX_API_KEY.strip()


def get_json(url, params, attempts=5):
    for attempt in range(attempts):
        try:
            response = session.get(url, params=params, timeout=60)

            if response.status_code == 429:
                time.sleep(5 * (attempt + 1))
                continue

            response.raise_for_status()
            return response.json()

        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(3 * (attempt + 1))


def make_abstract(index):
    if not index:
        return None

    words = []
    for word, positions in index.items():
        for pos in positions:
            words.append((pos, word))

    words.sort(key=lambda x: x[0])
    return " ".join(word for pos, word in words)


def normalize(text):
    return " ".join(str(text).lower().split())


# Ensure optional columns exist
extra_columns = [
    "openalex_id", "primary_topic", "subfield",
    "field", "domain", "topic_score", "source_url", "year"
]

for col in extra_columns:
    if col not in df.columns:
        df[col] = None

# Build sets from the existing 5,972 records
seen_ids = set(df["openalex_id"].dropna().astype(str))
seen_abstracts = set(
    df["abstract"].dropna().map(normalize)
)

new_records = []

def count_category(label):
    return int((df["label"] == label).sum()) + sum(
        row["label"] == label for row in new_records
    )


# Discover OpenAlex domains and fields
domains = get_json(
    f"{BASE_URL}/domains",
    {**api_params, "per_page": 100}
)["results"]

fields = get_json(
    f"{BASE_URL}/fields",
    {**api_params, "per_page": 100}
)["results"]

domain_ids = {
    d["display_name"]: d["id"].split("/")[-1]
    for d in domains
}

business_keywords = [
    "business",
    "management and accounting",
    "economics",
    "finance"
]

business_fields = [
    f for f in fields
    if any(
        keyword in f["display_name"].lower()
        for keyword in business_keywords
    )
]

business_field_ids = {
    f["id"].split("/")[-1] for f in business_fields
}


def collect_missing(label, filter_value):
    """Collect only enough new unique papers to reach 2,000."""

    count = count_category(label)

    if count >= TARGET:
        print(f"{label}: already has {count} papers.")
        return

    cursor = "*"
    print(f"\nFinding replacements for {label}: {count}/{TARGET}")

    while count < TARGET:
        params = {
            **api_params,
            "filter": (
                "has_abstract:true,language:en,type:article,"
                + filter_value
            ),
            "per_page": 100,
            "cursor": cursor,
            "select": (
                "id,doi,title,publication_year,"
                "abstract_inverted_index,primary_topic"
            )
        }

        data = get_json(f"{BASE_URL}/works", params)
        papers = data.get("results", [])

        if not papers:
            print("No more matching papers.")
            break

        for paper in papers:
            paper_id = paper.get("id")
            title = paper.get("title")
            abstract = make_abstract(
                paper.get("abstract_inverted_index")
            )

            if not paper_id or not title or not abstract:
                continue

            if len(abstract.split()) < 30:
                continue

            key = normalize(abstract)

            # Avoid both duplicate papers and duplicate abstract text
            if paper_id in seen_ids or key in seen_abstracts:
                continue

            topic = paper.get("primary_topic") or {}
            subfield = topic.get("subfield") or {}
            field = topic.get("field") or {}
            domain = topic.get("domain") or {}

            # Verify that the returned topic matches the class
            field_id = (
                field.get("id", "").split("/")[-1]
            )

            if label == "Science":
                if domain.get("display_name") not in [
                    "Physical Sciences",
                    "Life Sciences",
                    "Health Sciences"
                ]:
                    continue

            elif label == "Business":
                if field_id not in business_field_ids:
                    continue

            elif label == "Social Science":
                if domain.get("display_name") != "Social Sciences":
                    continue
                if field_id in business_field_ids:
                    continue

            new_records.append({
                "title": title.strip(),
                "abstract": abstract.strip(),
                "label": label,
                "source_url": paper.get("doi") or paper_id,
                "year": paper.get("publication_year"),
                "openalex_id": paper_id,
                "primary_topic": topic.get("display_name"),
                "subfield": subfield.get("display_name"),
                "field": field.get("display_name"),
                "domain": domain.get("display_name"),
                "topic_score": topic.get("score")
            })

            seen_ids.add(paper_id)
            seen_abstracts.add(key)
            count += 1

            if count >= TARGET:
                break

        # Save progress, including all existing data
        combined = pd.concat(
            [df, pd.DataFrame(new_records)],
            ignore_index=True
        )

        combined.to_csv(
            "research_abstracts_recovery_checkpoint.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print(f"{label}: {count}/{TARGET}")

        next_cursor = data.get("meta", {}).get("next_cursor")
        if not next_cursor or next_cursor == cursor:
            break

        cursor = next_cursor
        time.sleep(0.2)


# Fill only the categories that are short
for label in ["Science", "Business", "Social Science"]:

    if count_category(label) >= TARGET:
        continue

    if label == "Science":
        for domain_name in [
            "Physical Sciences",
            "Life Sciences",
            "Health Sciences"
        ]:
            if count_category(label) >= TARGET:
                break

            if domain_name in domain_ids:
                collect_missing(
                    label,
                    f"primary_topic.domain.id:{domain_ids[domain_name]}"
                )

    elif label == "Business":
        for field in business_fields:
            if count_category(label) >= TARGET:
                break

            field_id = field["id"].split("/")[-1]

            collect_missing(
                label,
                f"primary_topic.field.id:{field_id}"
            )

    else:
        domain_id = domain_ids.get("Social Sciences")

        if domain_id:
            collect_missing(
                label,
                f"primary_topic.domain.id:{domain_id}"
            )


# Combining the original data with newly collected records
df = pd.concat(
    [df, pd.DataFrame(new_records)],
    ignore_index=True
)

# Removing any remaining duplicate abstracts
df["abstract"] = df["abstract"].str.strip()
df["_normalized_abstract"] = df["abstract"].map(normalize)

df = (
    df.drop_duplicates("_normalized_abstract")
      .drop(columns="_normalized_abstract")
      .reset_index(drop=True)
)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nFINAL COUNTS")
print("Total unique abstracts:", len(df))
print(df["label"].value_counts())
print("\nSaved:", OUTPUT_FILE)



Finding replacements for Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 1994/2000
Science: 2000/2000

Finding replacements for Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1991/2000
Business: 1992/2000
Business: 2000/2000

Finding replacements for Social Science: 1987/2000
Social Science: 19

In [ ]:
# Final Check
for category in ["Science", "Business", "Social Science"]:
    count = int((df["label"] == category).sum())
    print(f"{category}: {count}/2000")

print("\nTotal:", len(df))
print("Unique abstract texts:", df["abstract"].map(
    lambda x: " ".join(str(x).lower().split())
).nunique())


Science: 2000/2000
Business: 2000/2000
Social Science: 2000/2000

Total: 6000
Unique abstract texts: 6000
